In [5]:
from pathlib import Path
import sys

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "utils").is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError("Project root with src/utils was not found")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("ROOT:", ROOT)


ROOT: f:\PycharmProjects\micro_ct


In [6]:
import torch

from utils import check_required_dependencies

status = check_required_dependencies()
print(status)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())


{'numpy': True, 'pandas': True, 'scipy': True, 'skimage': True, 'torch': True, 'porespy': True, 'openpnm': True, 'gudhi': True}
torch: 2.10.0+cu126
cuda available: True


In [ ]:
from utils import TopologyAdaptiveRoutedUNet3D, PoreNetworkPermeabilityModel

device = "cuda" if torch.cuda.is_available() else "cpu"

seg = TopologyAdaptiveRoutedUNet3D(base_channels=4, ctx_dim=16).to(device)
x = torch.randn(1, 1, 16, 16, 16, device=device)
ph = torch.randn(1, 6, device=device)
out = seg(x, ph_features=ph, return_dict=True)
logits = out["logits"]
print("seg logits:", tuple(logits.shape))

node_attr = torch.rand(3, 6, device=device)
edge_index = torch.tensor([[0, 1], [1, 2]], dtype=torch.long, device=device).t()
edge_attr = torch.rand(2, 3, device=device)
coords = torch.tensor([[0., 0., 0.], [0.5, 0., 0.], [1., 0., 0.]], device=device)
log_g_hp = torch.zeros(2, device=device)
gnn = PoreNetworkPermeabilityModel(6, 3, hidden=16, layers=2, mu=1.0).to(device)
k, log_g = gnn(node_attr, edge_index, edge_attr, coords, (1.0, 1.0, 1.0), log_g_hp)
print("k:", k.detach().cpu(), "log_g:", log_g.detach().cpu())


seg logits: (1, 1, 16, 16, 16)
k: tensor([0.0000, 0.0000, 0.5000]) log_g: tensor([0., 0.])


: 